<a href="https://colab.research.google.com/github/vanecornejo/Procesos-Estocasticos/blob/main/Actividad%205.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Ejercicio 3:** Aproximación de $P(t)$ mediante Uniformización

  Desarrollamos la aproximación numérica de la matriz de transición de probabilidad $P(t)$ para una Cadena de Markov en Tiempo Continuo (CMTC). Utilizaremos el método de **Uniformización de Jensen** (o aleatorización).

## Fórmula de Truncamiento
Para aproximar la serie infinita de Poisson, utilizaremos los primeros $M$ términos según la regla heurística optimizada:
$$M \approx \max \{rt + 5\sqrt{rt},\, 20\}$$

Donde la ecuación para cada matriz resultante está dada por:
$$P_M(t) = \sum_{k=0}^{M} \frac{e^{-rt} (rt)^k}{k!} \hat{P}^k$$


In [29]:
import numpy as np

In [30]:
# Definición de la matriz de tasas R del ejercicio 1
R = np.array([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
], dtype=float)

r = 6.0  # Tasa global tal que r >= max(r_i)

# Construcción analítica de la matriz estocástica P_hat
P_hat = np.array([
    [1/6, 1/3, 1/2, 0],
    [2/3, 0,   1/3, 0],
    [0,   1/3, 1/3, 1/3],
    [1/6, 0,   1/2, 1/3]
], dtype=float)

In [31]:
# Definimos una función

def calcular_P_t(t, r, P_hat):
    # Aplicamos la fórmula para determinar M
    M = int(np.ceil(max(r * t + 5 * np.sqrt(r * t), 20)))

    P_t = np.zeros_like(P_hat)
    P_hat_k = np.eye(P_hat.shape[0])  # Inicializa P_hat^0 como la matriz Identidad

    # Sumatoria de los primeros M términos (de k = 0 a M-1)
    for k in range(M):
        if k == 0:
            poisson_prob = np.exp(-r * t)
        else:
            poisson_prob = poisson_prob * (r * t) / k

        P_t += poisson_prob * P_hat_k
        P_hat_k = np.dot(P_hat_k, P_hat)  # Eleva la potencia de la matriz P_hat^(k+1)

    return P_t, M

In [32]:
# Hacemos la Ejecución del cálculo para t = 0.5, t = 1 y t = 5
P_05, M_05 = calcular_P_t(0.5, r, P_hat)
P_1, M_1 = calcular_P_t(1.0, r, P_hat)
P_5, M_5 = calcular_P_t(5.0, r, P_hat)

### Resultados de las Matrices de Transición $P(t)$

Para visualizar los valores numéricos de las aproximaciones de las matrices y la cantidad de términos $M$ utilizados en el truncamiento de la serie:

In [33]:
print(f"=== Matriz P(0.5) [M calculado = {M_05}] ===")
print(np.round(P_05, 5))

print(f"\n=== Matriz P(1.0) [M calculado = {M_1}] ===")
print(np.round(P_1, 5))

print(f"\n=== Matriz P(5.0) [M calculado = {M_5}] ===")
print(np.round(P_5, 5))

=== Matriz P(0.5) [M calculado = 20] ===
[[0.25061 0.21696 0.38666 0.14577]
 [0.25313 0.23836 0.37441 0.13409]
 [0.16912 0.19361 0.4203  0.21696]
 [0.15802 0.15744 0.39833 0.28621]]

=== Matriz P(1.0) [M calculado = 20] ===
[[0.20615 0.2039  0.39871 0.19124]
 [0.20828 0.20534 0.3979  0.18847]
 [0.19676 0.19838 0.40096 0.2039 ]
 [0.19205 0.194   0.40147 0.21248]]

=== Matriz P(5.0) [M calculado = 58] ===
[[0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]]


### Verificando la Ecuación de Chapman-Kolmogorov

La ecuación de Chapman-Kolmogorov establece que para cualquier tiempo intermedio se cumple la relación de paso:
$$P(1.0) = P(0.5) \times P(0.5)$$

Procedemos a evaluar de manera numérica el producto matricial de $P(0.5)$ por sí misma para contrastarlo con el cálculo real obtenido de $P(1.0)$.

In [34]:
# Realizar el producto interno de matrices
P_05_cuadrado = np.dot(P_05, P_05)

print("=== Producto Matricial P(0.5) * P(0.5) ===")
print(np.round(P_05_cuadrado, 5))

print("\n=== Matriz Original P(1.0) ===")
print(np.round(P_1, 5))

# Calcular la discrepancia o error absoluto máximo entre ambas matrices
error_absoluto = np.max(np.abs(P_1 - P_05_cuadrado))
print(f"\nDiferencia absoluta máxima: {error_absoluto:.2e}")

if error_absoluto < 1e-5:
    print("La ecuación de Chapman-Kolmogorov se verificó con éxito")
else:
    print("Hay una discrepancia numérica significativa.")

=== Producto Matricial P(0.5) * P(0.5) ===
[[0.20615 0.2039  0.39871 0.19124]
 [0.20828 0.20534 0.3979  0.18847]
 [0.19676 0.19838 0.40096 0.2039 ]
 [0.19205 0.194   0.40147 0.21248]]

=== Matriz Original P(1.0) ===
[[0.20615 0.2039  0.39871 0.19124]
 [0.20828 0.20534 0.3979  0.18847]
 [0.19676 0.19838 0.40096 0.2039 ]
 [0.19205 0.194   0.40147 0.21248]]

Diferencia absoluta máxima: 2.07e-06
La ecuación de Chapman-Kolmogorov se verificó con éxito


----
# **Ejercicio 4.2:** Algoritmo de Uniformización Adaptativo con Control de Tolerancia ($\epsilon$)

Vamos a implementar el algoritmo dinámico de uniformización. A diferencia del método anterior, este algoritmo calcula el número de términos $M$ de forma automática sobre la marcha, deteniendo el bucle en el momento exacto en que el error residual de la cola de Poisson es menor o igual a la tolerancia establecida: $\epsilon = 0.00001$.

### Criterio de Parada
El bucle se ejecuta mientras la masa de probabilidad acumulada de Poisson sea menor a $1 - \epsilon$:
$$\text{sum} = \sum_{k=0}^{M} \frac{e^{-rt} (rt)^k}{k!} < 1 - \epsilon$$

In [35]:
import numpy as np

In [36]:
# Definimos de la matriz de tasas R
R = np.array([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
], dtype=float)

In [37]:
# Calculamos r de forma dinámica (usando la igualdad del máximo)
r_i = np.sum(R, axis=1)
r = float(np.max(r_i))

In [38]:
# Calculamos la matriz estocástica P_hat
N = R.shape[0]
P_hat = np.zeros_like(R)
for i in range(N):
    for j in range(N):
        if i == j:
            P_hat[i, j] = 1.0 - (r_i[i] / r)
        else:
            P_hat[i, j] = R[i, j] / r

In [39]:
def tolerancia(R, t, epsilon, r, P_hat):
    N = R.shape[0]

    # 4. Inicialización de variables del algoritmo
    A = np.copy(P_hat)
    B = np.exp(-r * t) * np.eye(N)
    c = np.exp(-r * t)
    sum_val = c
    k = 1

    # 5. Bucle condicional basado en la tolerancia
    while sum_val < (1.0 - epsilon):
        c = c * (r * t) / k
        B = B + c * A
        A = np.dot(A, P_hat)
        sum_val = sum_val + c
        k = k + 1

    # Al finalizar el bucle, el número de términos M de la serie equivale a k - 1
    M = k - 1
    return B, M, sum_val

In [40]:
# Parámetro de tolerancia
epsilon = 0.00001

# Ejecutamos el algoritmo para los tres horizontes temporales
B_05, M_05, sum_05 = tolerancia(R, 0.5, epsilon, r, P_hat)
B_1, M_1, sum_1 = tolerancia(R, 1.0, epsilon, r, P_hat)
B_5, M_5, sum_5 = tolerancia(R, 5.0, epsilon, r, P_hat)

### Resultados e Identificación de $M$ por Caso

Inspeccionamos las matrices de probabilidad obtenidas mediante el criterio adaptativo y conocer cuántos términos reales ($M$) se requirieron en cada escenario temporal.

In [41]:
print(f"=== t = 0.5 (M requerido = {M_05}, Suma acumulada = {sum_05:.6f}) ===")
print(np.round(B_05, 5))

print(f"\n=== t = 1.0 (M requerido = {M_1}, Suma acumulada = {sum_1:.6f}) ===")
print(np.round(B_1, 5))

print(f"\n=== t = 5.0 (M requerido = {M_5}, Suma acumulada = {sum_5:.6f}) ===")
print(np.round(B_5, 5))

=== t = 0.5 (M requerido = 13, Suma acumulada = 0.999997) ===
[[0.25061 0.21696 0.38666 0.14577]
 [0.25313 0.23836 0.37441 0.13409]
 [0.16912 0.19361 0.4203  0.21696]
 [0.15802 0.15744 0.39833 0.28621]]

=== t = 1.0 (M requerido = 19, Suma acumulada = 0.999995) ===
[[0.20615 0.2039  0.39871 0.19124]
 [0.20828 0.20534 0.3979  0.18847]
 [0.19676 0.19838 0.40096 0.2039 ]
 [0.19205 0.194   0.40147 0.21248]]

=== t = 5.0 (M requerido = 56, Suma acumulada = 0.999993) ===
[[0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]
 [0.2 0.2 0.4 0.2]]


### Comparación de Rendimiento y Resultados

A continuación, construiremos una tabla comparativa automatizada para analizar la diferencia en el número de pasos ($M$) computados por la regla heurística fija del ejercicio anterior versus este nuevo algoritmo adaptativo por tolerancia.

In [42]:
# Calcular los valores teóricos del ejercicio anterior para comparar
M_fijo_05 = int(np.ceil(max(r * 0.5 + 5 * np.sqrt(r * 0.5), 20)))
M_fijo_1 = int(np.ceil(max(r * 1.0 + 5 * np.sqrt(r * 1.0), 20)))
M_fijo_5 = int(np.ceil(max(r * 5.0 + 5 * np.sqrt(r * 5.0), 20)))

print(f"{'Tiempo (t)':<12} | {'M (Regla Fija Anterior)':<25} | {'M (Algoritmo Tolerancia)':<25}")
print("-" * 68)
print(f"{0.5:<12} | {M_fijo_05:<25} | {M_05:<25}")
print(f"{1.0:<12} | {M_fijo_1:<25} | {M_1:<25}")
print(f"{5.0:<12} | {M_fijo_5:<25} | {M_5:<25}")

print("\nConclusión de la comparación:")
print("1. Los valores internos de las matrices P(t) son idénticos en ambos métodos a 5 decimales.")
print("2. El algoritmo por tolerancia es más eficiente en tiempos cortos (ej: para t=0.5 redujo de 20 a 13 términos),")
print("   evitando cálculos matriciales innecesarios mientras se garantiza matemáticamente el error.")

Tiempo (t)   | M (Regla Fija Anterior)   | M (Algoritmo Tolerancia) 
--------------------------------------------------------------------
0.5          | 20                        | 13                       
1.0          | 20                        | 19                       
5.0          | 58                        | 56                       

Conclusión de la comparación:
1. Los valores internos de las matrices P(t) son idénticos en ambos métodos a 5 decimales.
2. El algoritmo por tolerancia es más eficiente en tiempos cortos (ej: para t=0.5 redujo de 20 a 13 términos),
   evitando cálculos matriciales innecesarios mientras se garantiza matemáticamente el error.
